# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # `dataset.metadata` is a CroissantMetadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**All entity IDs below are shown using their `@id` as required by the dataset schema.**

In [ ]:
# List all record sets by @id and name
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets in the dataset:\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id} | name: {rs.name}")

# For each record set, list fields and their @id
for rs in record_sets:
    print(f"\nFields and columns for RecordSet '{rs.name}' (@id={rs.id}):")
    for field in rs.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | data type: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    Column @id: {col.id} | name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*All data entities are referenced by their `@id` as per guidelines.*

In [ ]:
# Extract data from each record set by @id
record_sets_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, choose the first record set
main_record_set_id = record_sets_ids[0]
print(f"Record columns for RecordSet @id '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
print(f"\nFirst few rows:\n", dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

*For demonstration, we show filtering and normalization on a numeric field and group by an attribute if present.*

In [ ]:
# Inspect available fields for the main record set
df = dataframes[main_record_set_id]

# Try to choose a numeric field, e.g. coverage, MW, or similar
# For this example, we'll try a set of likely field names (from mass spectrometry datasets)
possible_numeric_fields = ['Coverage', 'coverage',
                          'PeptideCounts', 'peptide_counts',
                          'MW', 'molecular_weight', 'MolecularWeight',
                          'pI', 'Abundance', 'abundance', 'NormalizedAbundance']
numeric_field = None
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field = field
        break
if numeric_field is None:
    # fallback to the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise ValueError("No numeric field found in DataFrame.")

print(f"Using numeric field: {numeric_field}\n")

# Set filter threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in RecordSet @id {main_record_set_id} with {numeric_field} > {threshold:.2f}:\n")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field (e.g. Sample, Condition, or similar)
possible_group_fields = ['Sample', 'sample', 'Group', 'Condition', 'condition']
group_field = None
for field in possible_group_fields:
    if field in df.columns:
        group_field = field
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo group field available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the selected numeric field (before filtering)
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='teal')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, boxplot by group
if group_field is not None:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the mass spectrometry dataset describing extracellular vesicles from human mast cells using the `mlcroissant` library,
referencing dataset entities by their `@id`. We reviewed schema structure, extracted records as DataFrames,
performed EDA including normalization and filtering of a numeric field, and visualized value distributions.

For deeper domain-specific analyses, consult the metadata and field documentation accessible through the Croissant schema.